# Kaggle 2x T4 - Qwen3.8-27B Uncensored (IQ4_XS) + upstream llama.cpp

Serves **qwen3.8-27b-uncensored-iq4xs** on `127.0.0.1:8080` behind an OpenAI-compatible API, with a
native **262,144-token** context window.

| | |
| --- | --- |
| Base | `Qwen/Qwen3.8-27B` (27B, hybrid attention, native 262K ctx) |
| Weights | Heretic-abliterated (`JonathanColetti/Qwen3.8-27B-Uncensored-GGUF`) |
| Quant | `IQ4_XS` - 15.3 GB, PPL 7.1583 vs 7.1557 f16 (**+0.0026**, inside noise) |
| Refusal rate | 12/100 harmful prompts, down from 98/100 for the base model |
| Runtime | **stock** llama.cpp - no PrismML fork needed |

Notes:

- **Upstream llama.cpp, not the PrismML fork.** This is a standard GGUF with standard
  quant types. The `PQ2_0`/`PTQ1_0` ternary types still need PrismML's build; this model
  does not. A recent build is still required for the `qwen35` architecture.
- **Why IQ4_XS and not IQ2_M.** IQ2_M is 4.7 GB smaller but costs +0.70 PPL, and the model
  card is explicit that the refusal boundary is the least stable property and that 2-bit
  compounds exactly that. Since the point of this model is its refusal behaviour,
  IQ4_XS is the correct pick: it is free in quality terms.
- **KV cache is quantized to `q8_0`** so 262K context fits in 32 GB. See the memory
  section at the end for the arithmetic.

## 1. Check the GPU

Run this first. You should see **two Tesla T4 GPUs**.

In [ ]:
!nvidia-smi

## 2. Build upstream llama.cpp with CUDA

Upstream `llama.cpp` is required for the `qwen35` architecture. A recent `main` is used
rather than a pinned commit, because MTP speculative decoding only landed in PR #22673
and an older build will silently ignore those tensors.

The CUDA build is the slow part of this boot; expect several minutes.

In [ ]:
import os
import subprocess
from pathlib import Path

LLAMA_CPP_DIR = Path("/kaggle/working/llama.cpp")
BIN_DIR = LLAMA_CPP_DIR / "build" / "bin"
LLAMA_SERVER_BIN = BIN_DIR / "llama-server"

if not LLAMA_SERVER_BIN.exists():
    if not LLAMA_CPP_DIR.exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "https://github.com/ggml-org/llama.cpp.git",
                str(LLAMA_CPP_DIR),
            ],
            check=True,
        )

    subprocess.run(
        [
            "cmake", "-B", "build",
            "-DGGML_CUDA=ON",
            "-DCMAKE_BUILD_TYPE=Release",
            "-DLLAMA_CURL=OFF",
        ],
        cwd=str(LLAMA_CPP_DIR),
        check=True,
    )
    subprocess.run(
        ["cmake", "--build", "build", "-j", "--target", "llama-server", "llama-cli"],
        cwd=str(LLAMA_CPP_DIR),
        check=True,
    )

if not LLAMA_SERVER_BIN.exists():
    raise RuntimeError("llama-server was not produced by the CUDA build")

print("llama-server:", LLAMA_SERVER_BIN)
print(subprocess.run([str(LLAMA_SERVER_BIN), "--version"],
                     capture_output=True, text=True).stdout.strip())

## 3. Download the IQ4_XS GGUF

This is the **Qwen3.8-27B-Uncensored-IQ4_XS.gguf** file (15.3 GB) from `JonathanColetti/Qwen3.8-27B-Uncensored-GGUF`. The fused build keeps the MTP
draft head inline, so speculative decoding is available later without a second download.

`IQ4_XS` was measured at PPL 7.1583 against a 7.1557 f16 baseline on wikitext-2 - a
0.0026 gap against a standard error of roughly 0.25, so it is not separable from full
precision.

In [ ]:
import subprocess

MODEL_DIR = Path("/kaggle/working/qwen38-uncensored")
MODEL_FILE = "Qwen3.8-27B-Uncensored-IQ4_XS.gguf"
MODEL_PATH = MODEL_DIR / MODEL_FILE

MODEL_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(["pip", "install", "-q", "-U", "huggingface_hub"], check=True)

from huggingface_hub import hf_hub_download

if not MODEL_PATH.exists():
    print("Downloading Qwen3.8-27B-Uncensored-IQ4_XS.gguf (15.3 GB)...")
    hf_hub_download(
        repo_id="JonathanColetti/Qwen3.8-27B-Uncensored-GGUF",
        filename=MODEL_FILE,
        local_dir=str(MODEL_DIR),
    )
else:
    print("Model already present")

size_gb = MODEL_PATH.stat().st_size / (1024 ** 3)
print(f"Model size: {size_gb:.2f} GB")
print(f"Model path: {MODEL_PATH}")

## 4. Start llama-server

Flags, and why each one is here:

| Flag | Value | Reason |
| --- | --- | --- |
| `-c` | `NUM_CTX` (262144) | The model's native context, not an extension |
| `-ngl` | `99` | Offload every layer; the T4 pair has room |
| `-fa` | `on` | Required for quantized KV cache to be usable |
| `-ctk` / `-ctv` | `q8_0` | Fits 262K context inside 32 GB |
| `-np` | `1` | Single user; one slot avoids KV duplication |
| `--alias` | `MTP_MODEL` | The name clients send |
| `--cache-ram` | `12288` | Reusable prefix cache for agentic clients |

The wait loop gives a cold 262K load room to finish, and exits early on a dead process so
a real crash does not sit out the whole budget.

In [ ]:
import json
import time
import urllib.request

LLAMA_URL = "http://127.0.0.1:8080"

# The name clients send. kaggle-rotate reads MTP_MODEL and NUM_CTX straight out of this
# notebook and reports the name from status; there is no config key for either.
MTP_MODEL = "qwen3.8-27b-uncensored-iq4xs"
NUM_CTX = 262144

# Where llama-server writes. kaggle-rotate's readiness gate tails this on failure.
SERVER_LOG = "/tmp/qwen38-llama-server.log"

# Offload every layer to the GPU pair.
NUM_GPU_LAYERS = 99
# Quantized KV cache. q8_0 is the default; q4_0 frees roughly another 4.5 GB if a boot
# ever comes up tight.
KV_CACHE_TYPE = "q8_0"
# Single-user server: one slot, so the KV cache is never duplicated per sequence.
NUM_SLOTS = 1
# Reusable prompt prefix cache in RAM. This is what keeps a multi-turn agentic client
# from re-prefilling its whole conversation on every request.
CACHE_RAM_MB = 12288

# MTP speculative decoding. The fused GGUF carries the draft head inline and measured
# 1.15-1.32x on an H200, but it needs llama.cpp PR #22673 and adds VRAM pressure at
# 262K context. Off by default: an unattended rotation should fail safe, not fast.
USE_MTP = False
MTP_DRAFT_N_MAX = 2

def server_ready():
    try:
        with urllib.request.urlopen(f"{LLAMA_URL}/health", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

if not server_ready():
    server_log_file = open(SERVER_LOG, "a")

    cmd = [
        str(LLAMA_SERVER_BIN),
        "-m", str(MODEL_PATH),
        "--alias", MTP_MODEL,
        "--host", "127.0.0.1",
        "--port", "8080",
        "-c", str(NUM_CTX),
        "-ngl", str(NUM_GPU_LAYERS),
        "-fa", "on",
        "-ctk", KV_CACHE_TYPE,
        "-ctv", KV_CACHE_TYPE,
        "-np", str(NUM_SLOTS),
        "--cache-ram", str(CACHE_RAM_MB),
    ]
    if USE_MTP:
        cmd += ["--spec-type", "draft-mtp", "--spec-draft-n-max", str(MTP_DRAFT_N_MAX)]

    print("$", " ".join(cmd), flush=True)

    # kaggle-rotate renders and runs this notebook unattended, so a cold 262K load has to
    # be given room. Bailing out on a dead process keeps a genuine crash from waiting out
    # the whole budget.
    llama_server_process = subprocess.Popen(
        cmd,
        stdout=server_log_file,
        stderr=subprocess.STDOUT,
    )

    for _ in range(1800):
        if server_ready():
            break
        if llama_server_process.poll() is not None:
            break
        time.sleep(1)

    if not server_ready():
        raise RuntimeError(
            f"llama-server did not become ready on {LLAMA_URL}. Check {SERVER_LOG}"
        )

print("llama-server is ready:", LLAMA_URL)
print("Model:", MTP_MODEL)
print("Context:", NUM_CTX)
print("KV cache:", KV_CACHE_TYPE)
print("Prefix cache:", f"{CACHE_RAM_MB} MB")

## 5. Verify the model

A real completion against the local endpoint, in thinking mode at `medium` reasoning
effort, which is what keeps the check short.

In [ ]:
payload = {
    "model": MTP_MODEL,
    "messages": [{"role": "user", "content": "Reply with exactly: Model is working."}],
    "stream": False,
    "temperature": 1.0,
    "top_p": 0.95,
    "top_k": 20,
    "min_p": 0.05,
    "max_tokens": 256,
    "reasoning_effort": "medium",
}

request = urllib.request.Request(
    f"{LLAMA_URL}/v1/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urllib.request.urlopen(request, timeout=600) as response:
    result = json.loads(response.read())

print(result["choices"][0]["message"]["content"].strip())
print("Qwen3.8-27B uncensored is serving on", LLAMA_URL)

## 6. Raw llama.cpp benchmark

Dropped from the rendered kernel by kaggle-rotate - see `DROP_MARKDOWN_MARKERS`.

In [ ]:
import subprocess

print(subprocess.run(
    [str(LLAMA_SERVER_BIN), "--version"],
    capture_output=True, text=True,
).stdout)

subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used",
     "--format=csv,noheader"],
    check=False,
)

print("server log tail:")
subprocess.run(["tail", "-n", "40", SERVER_LOG], check=False)

## 7. API benchmark - throughput and memory at 262K context

Dropped from the rendered kernel by kaggle-rotate.

The memory arithmetic this notebook depends on, at 262,144 tokens:

- Model, `IQ4_XS`: 15.3 GB
- KV cache: 16 full-attention layers (of 64; the other 48 are Gated DeltaNet and hold a
  fixed-size recurrent state that does not grow with context), 4 KV heads x 256 head dim.
  At `q8_0` that is ~8.5 GB; at `q4_0`, ~4.5 GB.
- Total at `q8_0`: ~24 GB, which fits the 32 GB T4 pair with headroom for compute buffers.

Dropping to `-ctk q4_0 -ctv q4_0` is the lever if a boot ever runs tight. Reducing
`NUM_CTX` is the other.

In [ ]:
import time

def timed(prompt, effort="low", max_tokens=256):
    payload = {
        "model": MTP_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "max_tokens": max_tokens,
        "reasoning_effort": effort,
    }
    request = urllib.request.Request(
        f"{LLAMA_URL}/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    start = time.time()
    with urllib.request.urlopen(request, timeout=900) as response:
        result = json.loads(response.read())
    return time.time() - start, result["usage"]

elapsed, usage = timed("Write a one-line Python hello-world program.")
print(f"{elapsed:.1f}s  {usage}")

subprocess.run(
    ["nvidia-smi", "--query-gpu=index,memory.total,memory.used",
     "--format=csv,noheader"],
    check=False,
)